# 01 · EDA — Multi-Store Sales Forecasting

Short exploratory pass over the Kaggle *Store Item Demand Forecasting* data
(daily `sales` for 10 stores × 50 items, 2013–2017).

> If the real Kaggle `train.csv` isn't downloaded yet, `ensure_raw_data()` writes a
> schema-identical **synthetic** file so this notebook still runs. See the README.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.utils import load_config
from src.data import ensure_raw_data, load_raw, data_is_synthetic

cfg = load_config('../config.yaml')
ensure_raw_data(cfg)
df = load_raw(cfg)
print('SYNTHETIC data' if data_is_synthetic(cfg) else 'REAL Kaggle data')
df.head()

## Shape, coverage, and integrity

In [ ]:
print('rows       :', f'{len(df):,}')
print('date range :', df.date.min().date(), '->', df.date.max().date())
print('stores     :', sorted(df.store.unique()))
print('items      :', df.item.nunique())
print('series     :', df[['store','item']].drop_duplicates().shape[0])
print('sales min/max/mean:', df.sales.min(), df.sales.max(), round(df.sales.mean(),2))
print('missing values    :', int(df.isna().sum().sum()))

## Sales distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8,3))
ax.hist(df.sales, bins=60, color='#1f77b4')
ax.set_title('Distribution of daily sales'); ax.set_xlabel('sales'); ax.set_ylabel('count')
plt.tight_layout(); plt.show()

## Yearly seasonality (mean sales by month)

In [ ]:
by_month = df.assign(month=df.date.dt.month).groupby('month').sales.mean()
fig, ax = plt.subplots(figsize=(8,3))
ax.plot(by_month.index, by_month.values, marker='o')
ax.set_title('Mean daily sales by calendar month'); ax.set_xlabel('month'); ax.set_ylabel('mean sales')
plt.tight_layout(); plt.show()

## Weekly pattern (mean sales by day of week)

In [ ]:
by_dow = df.assign(dow=df.date.dt.dayofweek).groupby('dow').sales.mean()
fig, ax = plt.subplots(figsize=(8,3))
ax.bar(by_dow.index, by_dow.values, color='#2ca02c')
ax.set_xticks(range(7)); ax.set_xticklabels(['Mon','Tue','Wed','Thu','Fri','Sat','Sun'])
ax.set_title('Mean daily sales by day of week'); ax.set_ylabel('mean sales')
plt.tight_layout(); plt.show()

## An example series over time (store 1, item 1)

In [ ]:
s = df[(df.store==1)&(df.item==1)].sort_values('date')
fig, ax = plt.subplots(figsize=(11,3))
ax.plot(s.date, s.sales, lw=0.6)
ax.plot(s.date, s.sales.rolling(28).mean(), color='red', label='28-day MA')
ax.set_title('store 1 · item 1 — daily sales'); ax.legend()
plt.tight_layout(); plt.show()

## Takeaways for modelling
- Clear **weekly** and **yearly** seasonality → calendar + cyclical features, and
  lag/rolling features on the shifted series.
- Strong per-series level differences → lag features carry the level; store/item kept
  as categoricals for XGBoost.
- No missing dates / no negative sales (asserted in `data.load_raw`).